# Carga y calidad — donde el pipeline audita los datos

Corre el ETL completo (`cargar_datos.py`): construye la base desde el DDL, vuelca los
4 archivos a Bronze con linaje, resuelve identidad de cliente/plan/canal, parsea
fechas y montos sucios, y carga Silver. Luego aplica las vistas Gold
(`sql/02_vistas_gold.sql`).

In [1]:
import subprocess, sys
ROOT = r"D:\Claudia\Alegra"

resultado = subprocess.run([sys.executable, "cargar_datos.py"], cwd=ROOT, capture_output=True, text=True)
print(resultado.stdout)
if resultado.returncode != 0:
    print(resultado.stderr)
    raise RuntimeError("cargar_datos.py fallo")

Bronze cargado: CRM 262 | Billing 286 | Contratos 49 | Marketing 72
dim_cliente: 235 empresas resueltas desde CRM (17 con mas de un nombre historico, 3 nombres ambiguos entre empresas distintas)
fact_evento_suscripcion: 283 eventos cargados, 3 rechazados a cuarentena
fact_contrato: 49 contratos cargados
fact_gasto_adquisicion: 72 filas cargadas, 0 rechazadas

PRAGMA foreign_key_check: 0 violaciones -> OK
Hallazgos de calidad: 15 (detalle en reporte_calidad.json)
Clientes con mas de un nombre historico (rebranding detectado): 17

Carga completa. Base: db/saas_metrics.db



In [2]:
import sqlite3
con = sqlite3.connect(f"{ROOT}/db/saas_metrics.db")
con.executescript(open(f"{ROOT}/sql/02_vistas_gold.sql", encoding="utf-8").read())
con.commit()
print("Vistas gold aplicadas.")

Vistas gold aplicadas.


## Lo que encontró el pipeline

Tres tipos de hallazgo, cada uno con una decisión distinta:

1. **Rebranding real** — CRM conoce más de un nombre para el mismo contacto. Se
   resuelve automáticamente (el ancla es el ID de CRM, no el nombre).
2. **Colisión de nombre** — dos empresas reales distintas comparten el mismo nombre
   normalizado. No se puede resolver automáticamente sin más información: se marca
   ambiguo y sus movimientos de Billing se aíslan como clientes huérfanos, cada uno
   por su propio `id_customer_stripe`.
3. **Contrato sin match confiable** — error de digitación demasiado severo para la
   normalización. Se marca huérfano en vez de forzar un match.

In [3]:
import json
import pandas as pd

reporte = json.load(open(f"{ROOT}/reporte_calidad.json", encoding="utf-8"))

print(f"Clientes con mas de un nombre historico en CRM (candidatos a rebranding): {len(reporte['alias_cliente_dificiles'])}")
pd.DataFrame(reporte["alias_cliente_dificiles"])[["id_contacto_crm", "nombres"]].head(10)

Clientes con mas de un nombre historico en CRM (candidatos a rebranding): 17


,id_contacto_crm,nombres
0,CRM-000003,"[GALINDO, GARCÍA AND SALAZAR, López and Sons]"
1,CRM-000024,"[Pardo, Álvarez and Hoyos, Loaiza-Franco]"
2,CRM-000025,"[Ortiz and Sons, Londoño, Muñoz and Orozco]"
3,CRM-000043,"[Hernández Group, MANTILLA PLC]"
4,CRM-000048,"[Torres LLC, FERNÁNDEZ AND SONS]"
5,CRM-000065,"[Leal Inc, Rivera Group]"
6,CRM-000075,"[Navarro-Ospina, Vargas, Hurtado and Beltrán]"
7,CRM-000076,"[Betancur-Ramírez, Álvarez, Ortiz and Hernández]"
8,CRM-000083,"[Suárez, Castrillón and Arévalo, Rodríguez, Ay..."
9,CRM-000095,"[Maldonado-Rosero, Cortés, Osorio and Gómez]"


In [4]:
hallazgos = pd.Series(reporte["hallazgos"])
colisiones = hallazgos[hallazgos.str.contains("AMBIGUO")]
sin_match = hallazgos[hallazgos.str.contains("SIN MATCH")]
print(f"Nombres marcados AMBIGUOS (colision real detectada): {len(colisiones)}")
for h in colisiones:
    print(" -", h)
print(f"\nRegistros sin match confiable (huerfanos): {len(sin_match)}")
for h in sin_match:
    print(" -", h)

Nombres marcados AMBIGUOS (colision real detectada): 3
 - Nombre 'RIVERA GROUP' usado por mas de una empresa distinta en CRM (id_cliente 16 y 65): se marca AMBIGUO y NO se cruza automaticamente contra Billing/Contratos - cada lado se resuelve por su identificador propio (id_customer_stripe / fila de contrato), nunca por este nombre
 - Nombre 'CASTILLO LLC' usado por mas de una empresa distinta en CRM (id_cliente 57 y 105): se marca AMBIGUO y NO se cruza automaticamente contra Billing/Contratos - cada lado se resuelve por su identificador propio (id_customer_stripe / fila de contrato), nunca por este nombre
 - Nombre 'GOMEZ-QUINTERO' usado por mas de una empresa distinta en CRM (id_cliente 90 y 153): se marca AMBIGUO y NO se cruza automaticamente contra Billing/Contratos - cada lado se resuelve por su identificador propio (id_customer_stripe / fila de contrato), nunca por este nombre

Registros sin match confiable (huerfanos): 12
 - SIN MATCH confiable en BILLING: 'Rivera Group  ' -> si

In [5]:
resumen_calidad = pd.read_sql_query("SELECT * FROM gold_v_resumen_calidad", con)
print("Resumen de calidad de dim_cliente:")
resumen_calidad

Resumen de calidad de dim_cliente:


,estado_calidad,num_clientes,mrr_inicial_cop
0,HUERFANO_O_AMBIGUO,12,1145000.0
1,OK,219,52935830.0
2,REBRANDING,16,4140050.0


In [6]:
cuarentena = pd.read_sql_query("SELECT * FROM gold_v_calidad_cuarentena", con)
print("Filas rechazadas a cuarentena, por fuente:")
cuarentena

Filas rechazadas a cuarentena, por fuente:


,fuente,filas_rechazadas,motivos
0,BILLING,3,tipo_evento invalido o vacio: 'nan'


## Verificación: ¿el pipeline reconstruye la verdad de control?

In [7]:
control = json.load(open(f"{ROOT}/data/control_totales.json", encoding="utf-8"))

q = """
SELECT id_periodo,
       SUM(CASE WHEN tipo_evento='nueva_suscripcion' THEN monto_mensual_cop ELSE 0 END) AS mrr_nuevo,
       SUM(CASE WHEN tipo_evento='cancelacion' THEN monto_mensual_cop ELSE 0 END) AS mrr_churn,
       SUM(CASE WHEN tipo_evento IN ('nueva_suscripcion','reactivacion') THEN 1 ELSE 0 END) AS altas,
       SUM(CASE WHEN tipo_evento='cancelacion' THEN 1 ELSE 0 END) AS bajas
FROM fact_evento_suscripcion GROUP BY id_periodo ORDER BY id_periodo
"""
recon = pd.read_sql_query(q, con).set_index("id_periodo")
verdad = pd.DataFrame(control["mensual"]).T
verdad.index = verdad.index.astype(int)

comparado = recon.join(verdad[["mrr_nuevo", "mrr_churn", "altas", "bajas"]], rsuffix="_verdad")
comparado["diff_altas"] = comparado["altas"] - comparado["altas_verdad"]
comparado["diff_mrr_nuevo"] = comparado["mrr_nuevo"] - comparado["mrr_nuevo_verdad"]
comparado[["altas", "altas_verdad", "diff_altas", "mrr_nuevo", "mrr_nuevo_verdad", "diff_mrr_nuevo"]]

,altas,altas_verdad,diff_altas,mrr_nuevo,mrr_nuevo_verdad,diff_mrr_nuevo
id_periodo,,,,,,
202401,28,28.0,0.0,8254340,8254340.0,0.0
202402,16,16.0,0.0,4141430,4141430.0,0.0
202403,24,26.0,-2.0,8962480,9460480.0,-498000.0
202404,16,16.0,0.0,4564430,4564430.0,0.0
202405,23,24.0,-1.0,8627000,8976000.0,-349000.0
202406,13,13.0,0.0,4294050,4294050.0,0.0
202407,18,18.0,0.0,6244240,6244240.0,0.0
202408,18,18.0,0.0,6552080,6552080.0,0.0
202409,22,22.0,0.0,6580830,6580830.0,0.0


## Advertencia metodológica

La reconstrucción **no cuadra al 100% y es correcto que no lo haga**: 3 filas de
Billing llegaron con `tipo_evento` vacío (perdido en el origen, no algo que se pueda
adivinar) y van a cuarentena, como corresponde. El efecto se propaga: si se pierde el
evento de alta de un cliente, cualquier upgrade/downgrade posterior de ese mismo
cliente pierde su "plan anterior" y no se puede clasificar como expansión o
contracción. El resultado es un desvío de **~1% del MRR** a diciembre, identificado y cuantificado — no oculto.

Un segundo efecto, menor, es de medición y no de pérdida de datos: el MRR de un cliente en USD queda registrado a la tasa del mes de su último evento, mientras su baja se valora a la tasa del mes en que cancela (en noviembre eso explica 13.110 COP). Detalle en `docs/INFORME.md`. Es la misma postura que un análisis riguroso
de calidad de datos exige: mostrar el hueco, no maquillarlo.